In [7]:
import sqlite3
import pandas as pd

con = sqlite3.connect("../data/raw/ipl.db")

def q(sql):
    return pd.read_sql(sql, con)

# 1, Find the top 10 grounds by number of matches.

In [8]:
q("""
SELECT
    venue,
    COUNT(DISTINCT match_id) AS matches_played
FROM matches
WHERE venue IS NOT NULL
GROUP BY venue
ORDER BY matches_played DESC
LIMIT 10;
""")

,venue,matches_played
0,Eden Gardens,77
1,Wankhede Stadium,73
2,M Chinnaswamy Stadium,65
3,Feroz Shah Kotla,60
4,"Wankhede Stadium, Mumbai",57
5,"Rajiv Gandhi International Stadium, Uppal",49
6,"MA Chidambaram Stadium, Chepauk",48
7,Sawai Mansingh Stadium,47
8,Dubai International Cricket Stadium,46
9,"MA Chidambaram Stadium, Chepauk, Chennai",38


# 2, Find grounds with an average innings score above 165, using a minimum of 25 matches.

In [9]:
q("""
SELECT
    venue,
    COUNT(*) AS innings_played,
    ROUND(AVG(innings_score), 2) AS avg_innings_score
FROM (
    SELECT
        m.venue,
        d.match_id,
        d.innings,
        SUM(d.total_runs) AS innings_score
    FROM matches m
    JOIN deliveries d
        ON m.match_id = d.match_id
    WHERE m.venue IS NOT NULL
    GROUP BY m.venue, d.match_id, d.innings
)
GROUP BY venue
HAVING COUNT(DISTINCT match_id) >= 25
   AND AVG(innings_score) > 165
ORDER BY avg_innings_score DESC;
""")

,venue,innings_played,avg_innings_score
0,"Arun Jaitley Stadium, Delhi",56,183.23
1,"Eden Gardens, Kolkata",53,182.21
2,"Narendra Modi Stadium, Ahmedabad",74,177.82
3,"Wankhede Stadium, Mumbai",114,175.58


# 3, Calculate the chase win percentage for grounds with at least 50 matches.

In [10]:
q("""
WITH second_innings AS (
    SELECT
        match_id,
        batting_team
    FROM deliveries
    WHERE innings = 2
    GROUP BY match_id, batting_team
)

SELECT
    m.venue,
    COUNT(DISTINCT m.match_id) AS matches_played,
    SUM(
        CASE
            WHEN m.match_winner = s.batting_team THEN 1
            ELSE 0
        END
    ) AS chase_wins,
    ROUND(
        SUM(
            CASE
                WHEN m.match_winner = s.batting_team THEN 1
                ELSE 0
            END
        ) * 100.0 / COUNT(DISTINCT m.match_id),
        2
    ) AS chase_win_percentage
FROM matches m
JOIN second_innings s
    ON m.match_id = s.match_id
WHERE m.venue IS NOT NULL
GROUP BY m.venue
HAVING COUNT(DISTINCT m.match_id) >= 50
ORDER BY chase_win_percentage DESC;
""")

,venue,matches_played,chase_wins,chase_win_percentage
0,Eden Gardens,77,47,61.04
1,"Wankhede Stadium, Mumbai",57,34,59.65
2,M Chinnaswamy Stadium,64,36,56.25
3,Feroz Shah Kotla,59,32,54.24
4,Wankhede Stadium,73,38,52.05


# 4, Count the number of unique cleaned venues.

In [11]:
q("""
SELECT
    COUNT(DISTINCT TRIM(venue)) AS unique_cleaned_venues
FROM matches
WHERE venue IS NOT NULL
  AND TRIM(venue) <> '';
""")

,unique_cleaned_venues
0,59


# 5, Find the five grounds with the lowest powerplay run rate

In [12]:
q("""
SELECT
    m.venue,
    ROUND(
        SUM(d.total_runs) * 1.0 /
        COUNT(
            DISTINCT d.match_id || '-' || d.innings || '-' || d.over_number
        ),
        2
    ) AS powerplay_run_rate
FROM matches m
JOIN deliveries d
    ON m.match_id = d.match_id
WHERE d.over_number BETWEEN 0 AND 5
  AND m.venue IS NOT NULL
GROUP BY m.venue
ORDER BY powerplay_run_rate ASC
LIMIT 5;
""")

,venue,powerplay_run_rate
0,OUTsurance Oval,5.58
1,Shaheed Veer Narayan Singh International Stadium,6.39
2,JSCA International Stadium Complex,6.46
3,Buffalo Park,6.58
4,Nehru Stadium,6.63


# 6, Explain why COUNT(DISTINCT match_id) can be safer than COUNT(*) after a JOIN.

### Why COUNT(DISTINCT match_id) is safer than COUNT(*) after a JOIN

After joining matches with deliveries, one match can appear in
many rows because every ball is stored as a separate delivery.

COUNT(*) counts all the delivery rows.

COUNT(DISTINCT match_id) counts each match only once.

Therefore, COUNT(DISTINCT match_id) is safer when we want to
count the actual number of matches after a JOIN.

# 7, Explain why the day/night question cannot be answered from match_date alone.

### Why day/night cannot be answered from match_date

The match_date column only tells us the date of the match.

It does not contain the match start time or a day/night indicator.

Therefore, we cannot determine whether a match was played during
the day or at night using match_date alone.

We would need additional information such as match start time
or a day/night indicator.